<a href="https://colab.research.google.com/github/lsgrep/serv/blob/main/notebooks/07_token_economics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7 — Token economics: pricing, routing, and the self-host call

**The claim you should be able to make when you finish:** *"I can price an AI
workload live, on a whiteboard, and land on a recommendation — including the
volume at which my recommendation flips."*

This is the lab for the question you will be asked in some form every time:
*what will this cost, and should we host it ourselves?* The failure mode is not
getting the arithmetic wrong. It is **hedging** — walking through considerations
without landing anywhere, or landing somewhere without naming what would change
your mind.

### Two rules for doing it out loud

1. **State assumptions first, round aggressively.** "Call it 20M input tokens a
   day, 5M output, so about 5 tokens per word — I'll round to $0.30 and $2.50
   per million." Precision you did not earn is worse than a stated approximation.
2. **Land on a call, then name the trigger.** "Managed, unequivocally — and I'd
   revisit at roughly 3x this volume, or the day residency becomes a
   requirement."

### The prices in this notebook are a snapshot you maintain

`servlab.pricing.MODELS` is a table with a `VERIFIED_ON` date, not something the
library knows. Vendor pages move monthly and intro rates expire. Volunteering
that fact — *"this is my August number, and the Flash intro pricing roughly
doubles in January, so let me show you both"* — is worth more than any discount.

In [ ]:
# Cell 1 — bootstrap. No GPU: this is all arithmetic and it belongs on a laptop.
REPO, BRANCH = "https://github.com/lsgrep/serv.git", "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)

from servlab import pricing as pr, napkin as nk, gateway as gw
from servlab.plots import use_style, SERIES, STATUS, bar_compare
use_style()

# Every number below is only as good as this line. Read it out loud.
print(pr.staleness())

## 1. The per-request spread is the architecture conversation

Same workload, every tier. The spread is not a detail — a 5x gap between tiers
is what makes routing worth building, and a 1.2x gap is what makes it a waste of
a sprint.

In [ ]:
IN_TOK, OUT_TOK = 1000, 300
CHATS_PER_MONTH = 10e6

rows = pr.compare(["gemini-3.1-pro", "claude-opus-5", "gpt-5.6-sol", "claude-sonnet-5",
                   "gemini-3.6-flash", "gemini-3.5-flash-lite", "gpt-5.6-luna",
                   "gemini-2.5-flash-lite"],
                  IN_TOK, OUT_TOK, requests_per_month=CHATS_PER_MONTH)
print(pr.format_compare(rows))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.4))
names = [r["model"] for r in rows]
costs = [r["per_month"] for r in rows]
colors = [SERIES[0] if r["tier"] == "cheap" else
          SERIES[1] if r["tier"] == "mid" else STATUS["critical"] for r in rows]
ax.barh(names[::-1], costs[::-1], color=colors[::-1], height=0.6)
ax.set_xscale("log")
ax.set_xlabel(f"$ per month at {CHATS_PER_MONTH/1e6:.0f}M requests")
ax.set_title(f"{IN_TOK:,} in / {OUT_TOK} out — cheap tier, mid tier, frontier")
for i, c in enumerate(costs[::-1]):
    ax.annotate(f"${c:,.0f}", xy=(c, i), xytext=(5, 0), textcoords="offset points",
                va="center", fontsize=9, color="#52514e")
ax.margins(x=0.25)
plt.show()

## 2. The three discounts, and the one surcharge

Four multipliers move a bill more than any model choice, and three of them are
free:

* **Batch tier: ~50% off.** Anything not user-facing — document pipelines,
  backfills, evals, enrichment — should be here. Ask "does a human wait for
  this?" and if not, batch it.
* **Context caching: reads at ~10% of input.** A shared system prompt or a fixed
  document that appears in every request is paid for once.
* **Prompt hygiene**: the tokens you do not send.
* **Thinking tokens: billed as output.** The surcharge. A reasoning model's
  hidden trace is charged at the output rate — 5-6x input — even though nobody
  reads it. This is the usual answer to *"why did the bill double when traffic
  didn't?"*

In [ ]:
base = pr.call_cost("gemini-3.1-pro", IN_TOK, OUT_TOK) * CHATS_PER_MONTH
variants = {
    "standard": base,
    "batch tier": pr.call_cost("gemini-3.1-pro", IN_TOK, OUT_TOK, batch=True) * CHATS_PER_MONTH,
    "800 of 1000 input\ntokens cached": pr.call_cost(
        "gemini-3.1-pro", IN_TOK - 800, OUT_TOK, cached_input_tokens=800) * CHATS_PER_MONTH,
    "reasoning:\n+2000 thinking tok": pr.call_cost(
        "gemini-3.1-pro", IN_TOK, OUT_TOK, thinking_tokens=2000) * CHATS_PER_MONTH,
}
for k, v in variants.items():
    print(f"  {k.replace(chr(10), ' '):<34}${v:>10,.0f}/mo   {v/base:>5.2f}x")

bar_compare(list(variants), list(variants.values()),
            title="same model, same traffic — four billing regimes",
            ylabel="$ / month", highlight={"reasoning:\n+2000 thinking tok"}, fmt="${:,.0f}")

### The CFO version

> "The model didn't change and traffic didn't change. We switched to a reasoning
> model, and reasoning models bill their internal deliberation at the output
> rate — which is about six times the input rate. We're paying for thinking
> nobody reads. The fix isn't to stop using it; it's to route to it only where
> the reasoning is the product, and put the rest on the cheap tier with batch
> and caching on. That's a two-week change and it takes the bill back under the
> old number."

## 3. Routing: where most of the money actually is

Send everything to the cheap tier, judge the answer, escalate what fails. The
important accounting detail: **an escalated request paid for the cheap attempt
too.** Estimates that ignore that come in optimistic and then get quoted.

Two numbers to have: the blended cost at your expected escalation rate, and the
**break-even rate** past which routing costs more than just using the strong
model. Knowing the second one stops a router being defended past its usefulness.

In [ ]:
curve = pr.routing_curve("gemini-3.5-flash-lite", "gemini-3.1-pro",
                         IN_TOK, OUT_TOK, CHATS_PER_MONTH)
be = pr.breakeven_escalation("gemini-3.5-flash-lite", "gemini-3.1-pro", IN_TOK, OUT_TOK)

for r in curve:
    print(f"  escalate {r['escalation_rate']:>5.0%}   ${r['per_month']:>10,.0f}/mo   "
          f"{r['vs_all_strong']:>5.0%} of all-frontier   saves ${r['saved_per_month']:>10,.0f}")
print(f"\nbreak-even escalation rate: {be:.0%}")

In [ ]:
import matplotlib.pyplot as plt

all_strong = pr.call_cost("gemini-3.1-pro", IN_TOK, OUT_TOK) * CHATS_PER_MONTH
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot([r["escalation_rate"] * 100 for r in curve],
        [r["per_month"] for r in curve], marker="o", color=SERIES[0], label="routed")
ax.axhline(all_strong, color=STATUS["critical"], linestyle=":", linewidth=1.5)
ax.annotate("everything on the frontier model", xy=(0, all_strong), xytext=(4, 6),
            textcoords="offset points", color=STATUS["critical"], fontsize=9)
ax.axvline(be * 100, color=STATUS["warning"], linestyle="--", linewidth=1.5)
ax.annotate(f"break-even {be:.0%}", xy=(be * 100, all_strong * 0.4), xytext=(-90, 0),
            textcoords="offset points", color=STATUS["warning"], fontsize=9)
ax.set_xlabel("% of traffic escalated to the frontier model")
ax.set_ylabel("$ per month")
ax.set_title("routing: most of the saving survives a high escalation rate")
ax.legend(loc="upper left")
plt.show()

The shape worth internalising: **the curve is flat early.** Even escalating 30%
of traffic keeps you near half the all-frontier bill. That is the argument for
shipping the router *before* you can predict the escalation rate — you do not
need to be right about it to capture most of the benefit.

And the counter-argument, which you should raise yourself: routing adds a
quality-variance surface and a judge to maintain. On a $5K/month bill, do not
bother. On a $500K/month bill, it is the highest-leverage week of work available.

## 4. Managed or self-hosted — the three workloads

Work each one the way you would at a whiteboard: assumptions, arithmetic, call,
trigger.

In [ ]:
# Workload A — internal assistant. 20M in / 5M out per day, low quality bar.
w = pr.WORKLOADS["internal-assistant"]
monthly = pr.daily_cost("gemini-3.5-flash-lite", w.input_tokens_per_day,
                        w.output_tokens_per_day) * 30
print(f"20M in + 5M out/day on the cheap tier: ${monthly/30:,.2f}/day  ->  ${monthly:,.0f}/month")
print(f"\nOne engineer-day (~$1,200) is {1200/monthly:.1f} months of this bill.")
print("Call: managed, cheapest tier, move on. There is no analysis to do —")
print("doing the analysis costs more than the workload.")

In [ ]:
# Workload B — document pipeline. 2B in / 500M out per day, batchable,
# a 70B open model passes the evals. This is the one worth arguing about.
w = pr.WORKLOADS["doc-pipeline"]

std = pr.daily_cost("gemini-3.5-flash-lite", w.input_tokens_per_day, w.output_tokens_per_day) * 30
batch = pr.daily_cost("gemini-3.5-flash-lite", w.input_tokens_per_day,
                      w.output_tokens_per_day, batch=True) * 30
print(f"managed, standard: ${std:,.0f}/mo")
print(f"managed, batch:    ${batch:,.0f}/mo   (nobody waits for a document pipeline)")

# Size the self-hosted fleet from measured throughput, at peak not average.
TOK_S_PER_NODE = 3500       # 8xH100 running a 70B in FP8 at healthy batch
nodes = pr.nodes_needed(w.output_tokens_per_day, TOK_S_PER_NODE, peak_factor=2.5)
print(f"\nself-host sizing: {w.output_tokens_per_day/86400:,.0f} tok/s average, "
      f"x2.5 peak / {TOK_S_PER_NODE:,} per node = {nodes:.1f} -> {int(nodes)+1} nodes")

In [ ]:
plan = pr.SelfHostPlan(nodes=int(nodes) + 1, gpus_per_node=8, usd_per_gpu_hour=2.50,
                       platform_fte=0.75, fte_loaded_monthly=30_000)
print(plan)
print()
result = pr.managed_vs_selfhost("gemini-3.5-flash-lite", w.input_tokens_per_day,
                                w.output_tokens_per_day, plan, batch=True)
print(pr.format_verdict(result, "flash-lite, batch tier"))

In [ ]:
# At what volume does the call flip? Draw it rather than asserting it.
import matplotlib.pyplot as plt

multiples = [0.5, 1, 2, 3, 5, 8, 12, 20]
managed, selfhost = [], []
for m in multiples:
    managed.append(pr.daily_cost("gemini-3.5-flash-lite", w.input_tokens_per_day * m,
                                 w.output_tokens_per_day * m, batch=True) * 30)
    n = int(pr.nodes_needed(w.output_tokens_per_day * m, TOK_S_PER_NODE, 2.5)) + 1
    selfhost.append(pr.SelfHostPlan(nodes=n, platform_fte=0.75 + 0.25 * (n > 8)).total_monthly)

fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.plot(multiples, managed, marker="o", color=SERIES[0], label="managed (batch tier)")
ax.plot(multiples, selfhost, marker="o", color=SERIES[1], label="self-hosted, fully loaded")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("volume, as a multiple of today"); ax.set_ylabel("$ per month")
ax.set_title("the crossover is the answer to \"when would you change your mind?\"")
ax.legend(loc="upper left")
plt.show()

cross = next((m for m, a, b in zip(multiples, managed, selfhost) if a > b), None)
print(f"crossover at roughly {cross}x today's volume" if cross else
      "managed wins across this entire range")
print("\nAnd the honest caveat: crossing it is necessary, not sufficient. You also")
print("take on the eval risk, the on-call, and the capacity planning — which is")
print("why the FTE line is in the model and why it should never be zero.")

## 5. What self-hosting actually costs that the invoice does not show

Say these unprompted when recommending managed — it is the difference between a
cost opinion and an engineering one:

* **The FTE is not optional.** Someone upgrades vLLM, chases a regression, and
  carries the pager. At 0.5-1 FTE loaded, that is $15-30K/month before a single
  GPU is switched on — and it is usually the largest line item at small scale.
* **You pay for reserved capacity idle.** Managed APIs are metered; a reserved
  cluster is not. A half-idle fleet at any per-GPU price is the real cost story,
  and utilisation is harder than it looks under bursty traffic.
* **You inherit the quality ceiling.** The frontier is rented, not bought. If
  the open model is 5 points behind on your eval, that is a product decision,
  not an infra one.
* **Model upgrades become projects.** On a managed API a better model is a
  string change. Self-hosted it is a migration.

And the cases where self-hosting is right regardless of the arithmetic: **data
residency or air-gap requirements**, **the customer already owns the GPUs and
the people**, **sustained volume far past the crossover**, or **a workload the
managed provider will not serve** (custom weights, unusual context, latency at
the edge).

## 6. Lock-in: price the exit instead of fearing it

The honest version, and one you can deliver while representing any vendor.
Name the real vectors, then compute what leaving would cost.

In [ ]:
for name, detail in gw.LOCKIN_VECTORS.items():
    print(f"* {name}\n    {detail}\n")

In [ ]:
exit_cost = gw.switching_cost(
    corpus_tokens=800e6,       # everything you would re-embed
    eval_cases=200,
    engineer_days=10,          # gateway swap, re-ingest, revalidate
    fine_tune_reruns=2, fine_tune_cost_each=8000,
    egress_gb=2000,
)
print(gw.format_switching_cost(exit_cost))

In [ ]:
# Compare that exit to a year of the workload it protects. Usually a small
# insurance premium against a large dependency — which is exactly how to
# frame it to an exec who has been told lock-in is an all-or-nothing choice.
annual = result["managed_monthly"] * 12
print(f"exit cost              ${exit_cost['total_usd']:>12,.0f}")
print(f"annual managed spend   ${annual:>12,.0f}")
print(f"exit as % of one year  {exit_cost['total_usd']/annual:>12.1%}")

### The sentences

> "Some lock-in is the price of value. The goal isn't zero dependency — it's a
> dependency we chose knowingly, with an exit cost we've measured. Ours is about
> $28K, mostly re-running two fine-tunes and ten engineer-days. That's under 5%
> of a year's spend, and I'd rather pay that than give up the quality."

> "Your eval suite is your exit option. If we can measure a model swap in an
> afternoon, vendor choice stays an economic decision instead of a hostage
> situation. I insist on this for customers precisely because it keeps *us*
> honest."

Note what the second one does: it is a genuinely pro-customer position that also
happens to be true, and saying it from inside a vendor raises your credibility
rather than costing you the deal.

## 7. Practise it live

The drill you will actually face: a workload description, spoken, and thirty
seconds before you should be talking. Run this cell for a random instance.

In [ ]:
from servlab import drills
d = drills.cost()

In [ ]:
d.reveal()

## What to be able to say afterwards

1. **Price a workload in under two minutes**, assumptions stated, rounded, landing
   on a number and a call.
2. **The four multipliers**: batch (-50%), caching (~10% of input on reads),
   prompt hygiene, and thinking tokens (billed as output — the one that bites).
3. **Routing economics**: most of the saving survives a high escalation rate,
   and there is a break-even past which routing is worse than not routing.
4. **The self-host comparison including people** — and that below roughly
   $20-30K/month of API spend, the humans cost more than the tokens.
5. **A measured exit cost**, so lock-in is a number rather than a fear.

**Next:** [lab 8](08_rag_and_evals.ipynb) builds the eval suite that all of
this depends on.